# **Atividade 3: Desenvolvimento de um Agente Criativo com IA Generativa**

## **1.0.1 - Introdução: Da Análise à Construção**

Nas atividades anteriores, meu foco foi a análise e a descoberta. Primeiro, eu treinei e avaliei modelos para prever o engajamento. Depois, na Atividade 2, usei clustering para investigar os padrões nos dados e descobri que o sucesso de um post não dependia só do tema, mas também da sua estrutura.

Nesta terceira atividade, meu objetivo mudou da análise para a construção. O desafio agora era usar tudo o que aprendi para construir um **Agente Inteligente** que funcionasse como um assistente criativo. Minha ideia era criar um sistema que usasse os modelos e as conclusões das outras atividades para ajudar um usuário a melhorar suas postagens, de forma prática e informada.

## **1.0.2 - A Arquitetura que Eu Projetei para o Agente**

Antes de começar a programar, eu precisava de um plano. Por isso, desenhei uma arquitetura modular para o agente, onde um "cérebro" central (um LLM) coordena um conjunto de ferramentas que eu mesmo criei. A inspiração para esse design veio diretamente das minhas conclusões da Atividade 2.

O fluxo de trabalho que eu implementei foi o seguinte:

1.  **Diagnóstico Inicial:** Primeiro, o agente recebe o post do usuário e usa a **Ferramenta 1 (Classificador de Engajamento)**, o melhor modelo que treinei na Atividade 1, para ter um veredito objetivo (`high` ou `low`).
2.  **Tomada de Decisão:**
    * Se o post já for bom (`high`), o agente parabeniza o usuário e encerra.
    * Se o post tiver potencial de melhoria (`low`), o agente inicia o processo criativo.
3.  **Ciclo de Melhoria:** O agente entra em um loop para tentar melhorar o post. Em cada tentativa, ele:
    * **Consulta o Relatório:** Usa o conhecimento que gerei na Atividade 2 (a análise de clusters com LLM) para entender o que funciona ou não para aquele tema específico.
    * **Gera uma Nova Versão:** Pede ao LLM para reescrever o post, usando o relatório como guia.
    * **Faz uma Autoavaliação:** Usa o classificador de novo para avaliar a própria sugestão.
    * **Tenta Novamente:** Se a sugestão ainda não for boa o suficiente, ele usa essa nova versão como ponto de partida para a próxima tentativa, aprendendo no processo.

Essa iteração contínua para tentar gerar o melhor post possível foi o caminho que encontrei para ter resultados bons com maior frequência, apesar de nem sempre o agente ser capaz de criar um post que o classificador da atividade 1 considere como high, mas essa é uma limitação mais relacionada com o meu MLP do que com o agente em si, se eu tivesse armado ele com modelos mais robustos e melhores do que os que eu utilizei nas atividades 1 e 2, tenho certeza que ele seria melhor, mas eu realmente acredito que seja muito mais relevante para essa atividade que eu conseguisse juntar tudo que eu fiz em um só lugar e ver como funcionaria.

## **1.1 - Preparando o Ambiente e Minhas Ferramentas**

### **1.1.1 - Instalações e Importações**
Para começar, preparei o ambiente de execução. Aqui eu instalei e importei todas as bibliotecas que precisaria para o projeto, como `pandas`, `joblib` e o cliente da `openai`.

In [ ]:
# instalações
!pip install -q pandas sentence-transformers scikit-learn joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.4 MB/s eta 0:00:00


In [ ]:
# importações
import joblib
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import openai
import time

In [ ]:
# instala o ollama no ambiente do colab
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


### **1.1.2 - Inicialização do Servidor LLM**
O cérebro do meu agente é o modelo `Llama 3.1`. Nesta etapa, eu garanto que o servidor local do Ollama está rodando e que o modelo está pronto para receber os meus comandos.

In [ ]:
# configuração e inicialização da llm
print("\niniciando o servidor ollama...")
!nohup ollama serve &
time.sleep(10)
print("baixando o modelo llama3.1...")
!ollama pull llama3.1
print("servidor e modelo da llm prontos.")


iniciando o servidor ollama...
nohup: appending output to 'nohup.out'
baixando o modelo llama3.1...

servidor e modelo da llm prontos.


### **1.1.3 - Carregando Meus Artefatos**
Neste ponto, eu carrego os arquivos que preparei nas atividades anteriores. Cada um deles funciona como uma ferramenta ou peça de conhecimento para o agente:
* **O Classificador:** Meu modelo da Atividade 1 para prever o engajamento.
* **O Clusterizador e o Padronizador:** O K-Means e o `StandardScaler` da Atividade 2, que usei para identificar o tema de novos posts.
* **O Relatório de Análise:** O arquivo `analise_clusters.csv` que eu criei com os resumos da LLM sobre cada cluster.
E algumas outras que eu acabei não utilizando.

In [ ]:
print("baixando as ferramentas e o relatório do agente...")

# relatório
!gdown --id 19j0odTTMrmosXzvTqmmOlZBtHt738zE9 -O analise_clusters.csv
# clusterizador (modelo_final da atividade 2)
!gdown --id 1xJnf6DT4XhuNnpQW944UnT3a29FFE5L0 -O agente_ferramenta_clusterizador.joblib
# padronizador (tambem da atividade 2)
!gdown --id 1QosG5A_1PruX6XEWdXEHsAZraxmeHsQS -O agente_ferramenta_padronizador.joblib
# classificador
!gdown 1XRtqjIhTqQvbZLT4KigSarm5yrSox4tg -O agente_ferramenta_classificador.joblib
# features estruturais
!gdown 1G-VyEevEad0ZNUoc3oH2KSCbc2HX7ccE -O agente_ferramente_features_estruturais.joblib
# modelo lookup
!gdown 1maDkCWniBw1TvPXHobeE8_3NqyVgGTII -O agente_ferramenta_lookup.joblib

print("download concluído.")


baixando as ferramentas e o relatório do agente...
/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=19j0odTTMrmosXzvTqmmOlZBtHt738zE9
To: /content/analise_clusters.csv
100% 7.98k/7.98k [00:00<00:00, 31.0MB/s]
/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1xJnf6DT4XhuNnpQW944UnT3a29FFE5L0
To: /content/agente_ferramenta_clusterizador.joblib
100% 239k/239k [00:00<00:00, 97.0MB/s]
/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to

In [ ]:
# carregamento de todas as ferramentas, são os modelos e informações obtidas nas atividades 1 e 2, achei interessante usar aqui.
print("\ncarregando ferramentas...")
clusterizador = joblib.load('agente_ferramenta_clusterizador.joblib')
padronizador_estrutural = joblib.load('agente_ferramenta_padronizador.joblib')
mapa_analise_clusters = pd.read_csv('analise_clusters.csv').set_index('cluster')
classificador_mlp = joblib.load('agente_ferramenta_classificador.joblib')
modelo_lookup = joblib.load('agente_ferramenta_lookup.joblib')

modelo_st = SentenceTransformer("all-MiniLM-L6-v2")
client_llm = openai.OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


carregando ferramentas...


## **1.2 - Definição dos Módulos do Agente**
Com o ambiente pronto, o próximo passo foi definir as funções que encapsulam a lógica de cada ferramenta. Eu separei o código em módulos com responsabilidades únicas para deixar o sistema mais organizado e fácil de entender.

É interessante dizer que essas funções não eram assim inicialmente, eu fui adicionando novas funcionalidades e depois reorganizei tudo nessa ordem para deixar o fluxo melhor.

In [ ]:
# função helper que centraliza a criação das features enriquecidas.
def processar_features_de_post(texto_do_post: str):
    dados_post = pd.DataFrame([texto_do_post], columns=['content'])
    dados_post['comprimento_texto'] = dados_post['content'].str.len()
    dados_post['tem_link'] = dados_post['content'].str.contains('http', case=False).astype(int)
    dados_post['n_hashtags'] = dados_post['content'].str.count('#')
    dados_post['n_mencoes'] = dados_post['content'].str.count('@')
    features_estruturais_post = dados_post[['comprimento_texto', 'tem_link', 'n_hashtags', 'n_mencoes']].values
    features_estruturais_padronizadas = padronizador_estrutural.transform(features_estruturais_post)
    embedding_texto_post = modelo_st.encode(dados_post['content'].tolist())
    x_enriquecido_post = np.hstack((embedding_texto_post, features_estruturais_padronizadas))
    return x_enriquecido_post

# avalia um post e prevê o engajamento ('high' ou 'low').
def prever_engajamento_post(texto_do_post: str):
    x_enriquecido_post = processar_features_de_post(texto_do_post)
    previsao_numerica = classificador_mlp.predict(x_enriquecido_post) #usa o MLP que treinei na atividade 1
    mapa_inverso = {1: 'high', 0: 'low'}
    return mapa_inverso[previsao_numerica[0]]

# llm para analisar, classificar o tema e reescrever o post.
def gerar_sugestao_criativa(texto_do_post: str):

    # prepara o contexto para a llm com o relatório
    contexto_clusters = ""
    for cluster_id, row in mapa_analise_clusters.iterrows():
        contexto_clusters += f"--- cluster id: {cluster_id} ---\n"
        contexto_clusters += f"tema geral: {row['tema_geral']}\n"
        contexto_clusters += f"resumo do que funciona (sucesso): {row['resumo_positivo']}\n"
        contexto_clusters += f"resumo do que não funciona (fracasso): {row['resumo_negativo']}\n\n"

    # promp principal
    system_prompt_mestre = """
    Você é um assistente de IA especialista em marketing de redes sociais. Sua tarefa é analisar um post de um usuário, fazer sugestões de melhora e dar um exemplo final de como ficaria o post com as suas sugestões.
    Você receberá um post e um relatório com vários temas (clusters) e análises sobre o que gera engajamento em cada um.
    Seu processo deve ser:
    1.  Leia o post do usuário.
    2.  Leia o relatório de temas e escolha o 'cluster id' cujo 'tema geral' melhor se encaixa com o post do usuário, informe-o do tema do cluster.
    3.  Use os resumos de 'sucesso' e 'fracasso' daquele cluster específico para sugerir mudanças (como links, tamanho do texto, sentimento e pontuação), focando em maximizar o engajamento
    4.  Explique, se necessário, o que o post do usuário faz errado e como ele poderia melhorar.
    5.  Use os resumos de 'sucesso' e 'fracasso' daquele cluster específico para reescrever o post do usuário, focando em maximizar o engajamento.
    6.  Sua resposta final deve ser um JSON contendo o id do cluster que você escolheu e a nova versão do post.

    Exemplo de saída:
    {
      "cluster_escolhido": ...,
      "tema_escolhido": "...",
      "sugestões_de_melhora: "...",
      "pontos_de_avaliação": "...",
      "post_melhorado": "..."
    }
    """

    # exemplos_few_shot = """
    # Exemplo 1:
    # Post do Usuário para Análise:
    # <post_usuario>
    # Qual a sua definição de felicidade?
    # </post_usuario>

    # Resposta JSON Ideal:
    # {
    #   "cluster_escolhido": 8,
    #   "tema_escolhido": "Desenvolvimento Pessoal e Motivação",
    #   "sugestoes_de_melhora": "O post original é muito vago e soa como uma pergunta genérica. Posts de sucesso neste tema costumam conectar uma observação pessoal a uma reflexão mais ampla, citar fontes ou ideias e terminar com uma pergunta aberta para incentivar uma discussão mais profunda.",
    #   "pontos_de_avaliação": "Positivo: O post agora tem uma narrativa e um gancho pessoal (a capa da revista), o que o torna mais identificável. A pergunta no final agora parece um convite genuíno para uma conversa, em vez de uma enquete.",
    #   "post_melhorado": "A capa desta revista Time me chamou a atenção algumas semanas atrás na fila do supermercado. Quem não ama emojis? Isso me lembrou da fórmula da felicidade de Scott Adams = saúde x liberdade. Saúde para que você possa aproveitar a vida, e liberdade para que você escolha fazer o que ama. Eu adicionaria mais dois: relacionamentos e serviço ao próximo. Relacionamentos com amigos e família, e serviço para permitir que você retribua e faça coisas além de si mesmo. Qual é a sua definição de felicidade?"
    # }

    # Exemplo 2:
    # Post do Usuário para Análise:
    # <post_usuario>
    # Novidade sobre saúde digital: aprovaram um inalador novo.
    # </post_usuario>

    # Resposta JSON Ideal:
    # {
    #   "cluster_escolhido": 1,
    #   "tema_escolhido": "Saúde Digital e Tecnologia em Saúde",
    #   "sugestoes_de_melhora": "A informação é boa, mas falta credibilidade e contexto. Posts de notícias que engajam bem geralmente incluem o link da fonte, usam hashtags relevantes para aumentar o alcance e adicionam uma frase de impacto ou opinião que resume a importância da notícia.",
    #   "pontos_de_avaliação": "Positivo: O post agora é mais informativo e confiável por causa do link da fonte. O hashtag #saudedigital aumenta a visibilidade e a frase final ('Um grande passo...') adiciona um toque de análise que pode iniciar uma conversa.",
    #   "post_melhorado": "FDA aprova inalador digital conectado a aplicativo https://engt.co/2SbOtxs #saudedigital Um grande passo para tornar os pacientes o ponto de cuidado."
    # }
    # """

    user_prompt_mestre = f"""
    Aqui está o relatório de temas e análises:
    <relatorio>
    {contexto_clusters}
    </relatorio>

    Aqui está o post do usuário que você deve analisar e melhorar:
    <post_usuario>
    {texto_do_post}
    </post_usuario>
    """

    # DEPOIS (com Few-Shot):
    user_prompt_mestre = f"""
    Aqui está o relatório de temas e análises:
    <relatorio>
    {contexto_clusters}
    </relatorio>

    Agora, com base no relatório, analise e melhore o seguinte post:
    <post_usuario>
    {texto_do_post}
    </post_usuario>
    """

    # --- passo 3: chamar a llm com a tarefa completa ---
    try:
        response = client_llm.chat.completions.create(
            model="llama3.1",
            messages=[
                {"role": "system", "content": system_prompt_mestre},
                {"role": "user", "content": user_prompt_mestre}
            ],
            temperature=0.5,
            response_format={"type": "json_object"}
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f'{{"erro": "falha ao chamar a llm: {e}"}}'

print("todas as ferramentas do agente estão prontas.")

todas as ferramentas do agente estão prontas.


### **1.3 - Módulo de Execução**

Com todas as minhas ferramentas prontas, criei a função principal que orquestra tudo. Ela é o ponto central do sistema. Aqui o agente faz uso de tudo que eu disponibilizei para ele para conseguir entregar a melhor resposta possível.
Eu implementei a lógica conforme expliquei lá em cima, mas vou repetir aqui.

* **Diagnóstico:** A primeira coisa que a função faz é usar o classificador para dar uma avaliação inicial no post do usuário.
* **Decisão:** Se o post já for bom (`high`), a função simplesmente parabeniza o usuário e encerra.
* **Ciclo de Melhoria:** Se o post for avaliado como `low`, a função entra em um modo criativo. Eu criei um loop onde o agente tem várias tentativas para melhorar o post. Em cada tentativa, ele:
    1.  Usa o LLM para gerar uma nova versão.
    2.  Avalia a própria sugestão usando o classificador.
    3.  Se a sugestão for boa, ele encerra o ciclo com sucesso.
    4.  Se não for, ele usa a sugestão recém-criada como ponto de partida para a *próxima* tentativa. Achei essa parte importante, pois força o agente a refinar suas ideias em vez de sempre começar do zero.
* **Relatório Final:** No final, a função apresenta a melhor sugestão que encontrou, seja ela uma versão com garantia de alto engajamento ou a melhor tentativa dentro dos limites que eu defini.

In [ ]:
import json
#  Executa o ciclo completo do agente criativo
def executar_agente_criativo(post_inicial: str):
    print(f"POST ORIGINAL:\n'{post_inicial}'")
    print("\n" + "="*50)

    print("AGENTE: Analisando o potencial de engajamento do seu post...")
    avaliacao_inicial = prever_engajamento_post(post_inicial)
    print(f"-> Avaliação Inicial: A previsão de engajamento para este post é '{avaliacao_inicial}'.")

    # decide se precisa melhorar o post ou não
    if avaliacao_inicial == 'high':
        print("\nAGENTE: Seu post já está excelente, parabéns!")
        return
    else:
        print("\nAGENTE: Preparando sugestões para melhorar o seu post...")

        post_atual = post_inicial
        sugestao_final = ""
        sucesso = False

        # tenta 7x, mas daria para tentar mais, só mudar o range
        for tentativa in range(7):
            print(f"\n{tentativa + 1}) Sugestão de Melhora: ")

            resposta_agente_str = gerar_sugestao_criativa(post_atual)

            try:
                # Carrega o JSON só dessa vez
                analise_llm = json.loads(resposta_agente_str)
                sugestao_atual = analise_llm.get('post_melhorado', '')

                if not sugestao_atual:
                    print("-> Seguestão da LLM foi inválida, tentando novamente...")
                    continue
            except json.JSONDecodeError:
                print("-> Formato de sugestão inválido, tentando novamente...")
                continue

            # Extrai as outras informações do objeto JSON já carregado
            tema_geral = analise_llm.get('tema_escolhido', 'N/A')
            print(f"-> Tema Gerado: '{tema_geral}'")
            sugestao_melhora = analise_llm.get('sugestoes_de_melhora', 'N/A')
            print(f"-> Sugestão de Melhora Gerada: '{sugestao_melhora}'")

            print(f"-> Nova Sugestão Gerada: '{sugestao_atual}'")
            print("-> Avaliando a nova sugestão...")
            avaliacao_da_sugestao = prever_engajamento_post(sugestao_atual)
            print(f"-> Avaliação da Sugestão: '{avaliacao_da_sugestao}'")

            # para garantir que não vai mandar uma versão tão ruim quanto a original, verifica a propria solução
            # e atualiza, se ainda estiver ruim, o post para a ultima versão melhorada, assim ele fica tentando melhorar
            # a ultima sugestão e não o texto original
            if avaliacao_da_sugestao == 'high':
                print("-> Encontrei o formato certo")
                sugestao_final = sugestao_atual
                sucesso = True
                break
            else:
                print("-> Essa versão ainda pode ser melhorada, tentando novamente...")
                post_atual = sugestao_atual
                sugestao_final = sugestao_atual

    # apresenta o resultado final
    print("\n" + "="*55)
    if sucesso:
        print("AGENTE: Consegui! Aqui está uma versão melhorada com alto potencial de engajamento:")
    else:
        print("AGENTE: nenhuma das minhas abordagens garante que seu post será popular, mas a melhor delas foi essa: ")

    print(f"\nSUGESTÃO FINAL:\n'{sugestao_final}'")
    print("="*55)

## **1.4 - Simulação:**
Esta é a célula onde eu demonstro o agente em ação. Eu defini um post inicial que eu sabia ter baixo potencial de engajamento para observar o agente executar seu ciclo de diagnóstico e melhoria. Meu objetivo aqui era validar na prática a arquitetura que eu projetei e ver se o agente conseguiria, de fato, transformar um conteúdo ruim em uma versão com alta chance de sucesso.

In [ ]:
# deixei duas opções, escrever o post na hora direto na caixinha de texto ou deixar aqui manualmente

# post diretamente no código
post_do_usuario = 'eu fui reprovado na prova da OAB, não sei mais o que fazer'


# só descomentar essa linha abaixo para rodar direto.
# post_do_usuario = input("Digite seu post aqui: ")

In [ ]:
# Executa o agente com o post definido na célula anterior
executar_agente_criativo(post_do_usuario)

POST ORIGINAL:
'eu fui reprovado na prova da OAB, não sei mais o que fazer'

AGENTE: Analisando o potencial de engajamento do seu post...
-> Avaliação Inicial: A previsão de engajamento para este post é 'low'.

AGENTE: Preparando sugestões para melhorar o seu post...

1) Sugestão de Melhora: 
-> Tema Gerado: 'Notícias e Opiniões'
-> Sugestão de Melhora Gerada: 'N/A'
-> Nova Sugestão Gerada: 'Eu fui reprovado na prova da OAB, mas isso não significa que eu desisti! Estou aprendendo com os meus erros e estou determinado a superar essa barreira. Quem já passou por algo semelhante? Compartilhe suas histórias de sucesso e ajudem-me a manter a motivação!'
-> Avaliando a nova sugestão...
-> Avaliação da Sugestão: 'low'
-> Essa versão ainda pode ser melhorada, tentando novamente...

2) Sugestão de Melhora: 
-> Tema Gerado: 'Produtividade e Motivação'
-> Sugestão de Melhora Gerada: 'N/A'
-> Nova Sugestão Gerada: 'Eu fui reprovado na prova da OAB, mas isso não significa que eu desisti! Estou apre

In [ ]:
post_do_usuario = 'Eu estou desempregado há 10 meses, o mercado está saturado e eu não tenho oportunidades'
executar_agente_criativo(post_do_usuario)

POST ORIGINAL:
'Eu estou desempregado há 10 meses, o mercado está saturado e eu não tenho oportunidades'

AGENTE: Analisando o potencial de engajamento do seu post...
-> Avaliação Inicial: A previsão de engajamento para este post é 'low'.

AGENTE: Preparando sugestões para melhorar o seu post...

1) Sugestão de Melhora: 
-> Tema Gerado: 'Carreira e Negócios'
-> Sugestão de Melhora Gerada: 'N/A'
-> Nova Sugestão Gerada: 'Eu estou desempregado há 10 meses e estou determinado a usar essa oportunidade para aprender e se desenvolver profissionalmente. Quero compartilhar minhas experiências e habilidades com vocês e saber como podem me ajudar a alcançar meus objetivos.'
-> Avaliando a nova sugestão...
-> Avaliação da Sugestão: 'low'
-> Essa versão ainda pode ser melhorada, tentando novamente...

2) Sugestão de Melhora: 
-> Tema Gerado: 'Carreira e Negócios'
-> Sugestão de Melhora Gerada: 'N/A'
-> Nova Sugestão Gerada: 'Estou desempregado há 10 meses e estou determinado a usar essa oportunida

## **1.5 - Conclusão Final**

Ao final, a implementação do agente criativo cumpriu o objetivo que eu estabeleci para esta atividade. Eu consegui construir um sistema funcional que não apenas gera texto, mas faz isso utilizando os classificadores e informações coletadas nas atividades anteriores, mesmo que em um notebook completamente diferente.

Acho que, apesar de simples, minha solução foi bem satisfatória, ela fechou a trajetória de compreensão dos modelos clássicos, uniu essa classificação com as informações que coletei a partir de um modelo não supervisionado e, finalmente, agora é capaz de fazer sugestões iterativamente com base no que os outros passos forneceram. Com isso em mente, finalizo aqui a última atividade da disciplina de Inteligência Artificial, mas estarei estudando para que eu consiga fazer algo muito melhor no futuro com base no que aprendi nesse processo.